## Day 1

**Task:** Build a hybrid content-based movie recommendation system and design a fine-grained sentiment classifier that aggregates user reviews into pros/cons summaries.


In [ ]:
# DAY 1: Hybrid recommendation + fine-grained sentiment

!pip -q install textblob

import pandas as pd
import numpy as np
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.DataFrame([
    [1,"Interstellar","Sci-Fi Adventure","space science future family emotional",8.7],
    [2,"Inception","Sci-Fi Thriller","dream mind heist action mystery",8.8],
    [3,"The Dark Knight","Action Crime","superhero crime action thriller intense",9.0],
    [4,"La La Land","Romance Musical","romance music love emotional artistic",8.0],
    [5,"The Martian","Sci-Fi Adventure","space survival science humor",8.0],
    [6,"Toy Story","Animation Family","animation friendship family comedy",8.3],
    [7,"Coco","Animation Family","music family animation emotional culture",8.4],
    [8,"The Notebook","Romance Drama","romance love emotional relationship",7.8],
    [9,"Avengers: Endgame","Action Sci-Fi","superhero action adventure friendship emotional",8.4],
    [10,"The Grand Budapest Hotel","Comedy Drama","comedy artistic adventure quirky",8.1]
], columns=["movie_id","title","genre","tags","rating"])

reviews = pd.DataFrame([
    ["Interstellar","The visuals are amazing and the science is fascinating.","positive","visuals"],
    ["Interstellar","The family story is touching and emotional.","positive","family"],
    ["Interstellar","The middle part feels a little slow.","negative","pacing"],
    ["Inception","Very clever story with excellent mind-bending ideas.","positive","story"],
    ["Inception","The action and visuals are impressive.","positive","visuals"],
    ["Inception","Some parts are confusing on the first watch.","negative","clarity"],
    ["The Dark Knight","Great acting and intense action.","positive","acting"],
    ["The Dark Knight","The villain is outstanding.","positive","acting"],
    ["The Dark Knight","A few scenes are quite dark.","negative","visuals"],
    ["La La Land","Beautiful music and wonderful performances.","positive","music"],
    ["La La Land","The romance is emotional and memorable.","positive","romance"],
    ["La La Land","The ending may feel sad.","negative","ending"],
    ["The Martian","Funny, smart and inspiring survival story.","positive","story"],
    ["The Martian","The science details are interesting.","positive","science"],
    ["The Martian","Some technical scenes are lengthy.","negative","pacing"],
    ["Toy Story","Fun, warm and perfect for families.","positive","family"],
    ["Toy Story","The characters are lovable.","positive","characters"],
    ["Coco","Beautiful music and a touching family story.","positive","music"],
    ["Coco","Very emotional and visually beautiful.","positive","visuals"],
    ["The Notebook","A romantic and emotional love story.","positive","romance"],
    ["The Notebook","It is quite predictable.","negative","story"],
    ["Avengers: Endgame","Amazing action and emotional character moments.","positive","action"],
    ["Avengers: Endgame","Great payoff for superhero fans.","positive","story"],
    ["The Grand Budapest Hotel","Funny, stylish and visually unique.","positive","visuals"],
    ["The Grand Budapest Hotel","The quirky characters are entertaining.","positive","characters"]
], columns=["title","review","sentiment","aspect"])

vectorizer = TfidfVectorizer(stop_words="english")
movie_matrix = vectorizer.fit_transform(movies["genre"] + " " + movies["tags"])
similarity = cosine_similarity(movie_matrix)

def fine_grained_sentiment(review):
    polarity = TextBlob(review).sentiment.polarity
    label = "Positive" if polarity > 0.15 else "Negative" if polarity < -0.15 else "Neutral"
    aspect_words = {
        "story":["story","plot","ending"], "acting":["acting","performance","performances","characters","villain"],
        "visuals":["visuals","visual","style"], "music":["music"], "pacing":["slow","lengthy","pacing"],
        "family":["family"], "romance":["romance","love"], "science":["science"], "action":["action"]
    }
    aspect = "general"
    low = review.lower()
    for a, words in aspect_words.items():
        if any(w in low for w in words):
            aspect = a
            break
    return aspect, label, round(polarity,3)

def review_summary(title):
    r = reviews[reviews["title"] == title]
    pos, neg = r[r.sentiment=="positive"], r[r.sentiment=="negative"]
    return {
        "positive_reviews": len(pos),
        "negative_reviews": len(neg),
        "pros": ", ".join(pos.aspect.unique()),
        "cons": ", ".join(neg.aspect.unique())
    }

print(fine_grained_sentiment("The visuals are beautiful but the story is a little slow."))
print(review_summary("Interstellar"))
